# 02 · Validation

Interactive companion to `pipeline/09_validate.py`, which writes `docs/validation_report.md`. It loads the processed tables, plots the Sentinel-1 baseline against NISAR, and shows the 2025→2026 change.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pipeline.utils import sar, geo

In [ ]:
st = pd.read_csv(ROOT/'data/processed/status.csv', dtype={'month': str})
m = pd.read_csv(ROOT/'data/processed/monthly_area.csv', dtype={'month': str})
metrics = json.load(open(ROOT/'data/processed/validation_metrics.json'))
{k: (v.get('status'), v.get('pass')) for k, v in metrics.items() if isinstance(v, dict)}

In [ ]:
rid = 'MAR_0001'
fig, ax = plt.subplots(figsize=(11, 4))
for sensor, g in m[m.reservoir_id == rid].groupby('sensor'):
    ax.plot(pd.to_datetime(g.month), g.area_km2, 'o-' if sensor == 'NISAR' else '-', ms=4, label=sensor)
s = st[st.reservoir_id == rid]
ax.fill_between(pd.to_datetime(s.month), s.p10_km2, s.p90_km2, alpha=.15, label='p10–p90')
ax.plot(pd.to_datetime(s.month), s.normal_km2, '--', label='normal'); ax.legend(); ax.set_ylabel('km²');

In [ ]:
pd.DataFrame(metrics['temporal']['rows'])

In [ ]:
pd.DataFrame(metrics['sensors'].get('rows', []))

In [ ]:
pd.DataFrame(metrics['s2'].get('rows', []))